# Analysis Task

This notebook performs a descriptive analysis of structural economic changes across 17 global economies using the World Bank data.

**Analysis Task Objectives:**
* **Compute Shares:** Calculate Industry and Manufacturing value-added as a share of GDP.
* **Compute Growth (CAGR):** Evaluate 2000–2025 growth trends for Industry, Manufacturing, and real GDP.
* **Summary Table:** Generate and export a consolidated `summary_table.csv`.
* **Data Interpretation:** Analyze signs of de-industrialization and compare structural shifts between advanced and emerging economies.

---


In [2]:
import tomllib
import pandas as pd
import numpy as np

# 1. READ CONFIG
with open("config.toml", "rb") as f:
    config = tomllib.load(f)

start_year = config["time"]["start_year"]
end_year = config["time"]["end_year"]
ind_key = config["series"]["industry_value_added_usd_const"]
man_key = config["series"]["manufacturing_value_added_usd_const"]
gdp_key = config["series"]["gdp_usd_real"]


# 2. READ LOCAL CSV
try:
    df_wide = pd.read_csv("output_wide.csv")
except FileNotFoundError:
    print("Error: output_wide.csv not found. Run main.py first.")
    exit(1)


# 3. HELPER
def get_value_with_fallback(series_row, target_year, min_year):
    # Backward scanning to handle missing observations
    if series_row is None:
        return None, np.nan
    for year in range(target_year, min_year - 1, -1):
        col_name = f"year_{year}"
        if col_name in series_row.index:
            value = series_row[col_name]
            if pd.notna(value):
                return year, value
    return None, np.nan

def calculate_share(value_added, gdp):
    # Compute value-added shares
    if pd.notna(value_added) and pd.notna(gdp) and gdp > 0:
        return value_added / gdp
    return np.nan

def calculate_cagr(start_value, end_value, start_year, end_year):
    # Compute long-term CAGR
    if (
    pd.notna(start_value)
    and pd.notna(end_value)
    and start_value > 0
    and end_year > start_year
    ):
        return (end_value / start_value) ** (1 / (end_year - start_year)) - 1
    return np.nan


# 4. DATA PROCESSING
summary_rows = []
for country_code in df_wide['country'].unique():
    c_df = df_wide[df_wide['country'] == country_code].set_index('indicator')
    row = {"country": country_code}

    ind_row = c_df.loc[ind_key] if ind_key in c_df.index else None
    man_row = c_df.loc[man_key] if man_key in c_df.index else None
    gdp_row = c_df.loc[gdp_key] if gdp_key in c_df.index else None
    
    # Start year value-added shares (Year 2020)
    _, ind_start_value = get_value_with_fallback(ind_row, start_year, start_year)
    _, man_start_value = get_value_with_fallback(man_row, start_year, start_year)
    _, gdp_start_value = get_value_with_fallback(gdp_row, start_year, start_year)

    row[f"industry_share_{start_year}"] = calculate_share(ind_start_value, gdp_start_value)
    row[f"manufacturing_share_{start_year}"] = calculate_share(man_start_value, gdp_start_value)
    
    # End year value-added shares (Year 2025: returns NaN)
    v_ind_end = ind_row[f"year_{end_year}"] if ind_row is not None and f"year_{end_year}" in ind_row.index else np.nan
    v_man_end = man_row[f"year_{end_year}"] if man_row is not None and f"year_{end_year}" in man_row.index else np.nan
    v_gdp_end = gdp_row[f"year_{end_year}"] if gdp_row is not None and f"year_{end_year}" in gdp_row.index else np.nan

    row[f"industry_share_{end_year}"] = calculate_share(v_ind_end, v_gdp_end)
    row[f"manufacturing_share_{end_year}"] = calculate_share(v_man_end, v_gdp_end)

    # Dynamic fallback end year tracking (finds latest valid data)
    latest_gdp_year, latest_gdp_value = get_value_with_fallback(gdp_row, end_year, start_year + 1)
    latest_ind_year, latest_ind_value = get_value_with_fallback(ind_row, end_year, start_year + 1)
    latest_man_year, latest_man_value = get_value_with_fallback(man_row, end_year, start_year + 1)

    ## Latest valid year for industry share
    gdp_matching_ind = gdp_row[f"year_{latest_ind_year}"] if gdp_row is not None and latest_ind_year and f"year_{latest_ind_year}" in gdp_row.index else np.nan
    if latest_ind_year:
        row[f"industry_share_{latest_ind_year}"] = calculate_share(latest_ind_value, gdp_matching_ind)
   
    ## Latest valid year for manufacturing share
    gdp_matching_man = gdp_row[f"year_{latest_man_year}"] if gdp_row is not None and latest_man_year and f"year_{latest_man_year}" in gdp_row.index else np.nan
    if latest_man_year:
        row[f"manufacturing_share_{latest_man_year}"] = calculate_share(latest_man_value, gdp_matching_man)

    # CAGR calculations
    row[f"gdp_cagr_{start_year}_{latest_gdp_year}"] = calculate_cagr(gdp_start_value, latest_gdp_value, start_year, latest_gdp_year) if latest_gdp_year else np.nan
    row[f"industry_cagr_{start_year}_{latest_ind_year}"] = calculate_cagr(ind_start_value, latest_ind_value, start_year, latest_ind_year) if latest_ind_year else np.nan
    row[f"manufacturing_cagr_{start_year}_{latest_man_year}"] = calculate_cagr(man_start_value, latest_man_value, start_year, latest_man_year) if latest_man_year else np.nan
    
    summary_rows.append(row)


# 5. EXPORT RESULTS
summary_table = pd.DataFrame(summary_rows)

# Print the summary table
print("\n--- SUMMARY TABLE PREVIEW ---")
print(summary_table.to_string(index=False))

# Save the csv file
summary_table.to_csv("summary_table.csv", index=False, float_format="%.6f")
print("Successfully generated summary_table.csv")


--- SUMMARY TABLE PREVIEW ---
country  industry_share_2000  manufacturing_share_2000  industry_share_2025  manufacturing_share_2025  industry_share_2024  manufacturing_share_2024  gdp_cagr_2000_2024  industry_cagr_2000_2024  manufacturing_cagr_2000_2024  manufacturing_share_2015  manufacturing_cagr_2000_2015  industry_share_2023  manufacturing_share_2023  industry_cagr_2000_2023  manufacturing_cagr_2000_2023  industry_share_2015  industry_cagr_2000_2015
    AUS             0.235576                  0.090724                  NaN                       NaN             0.203457                  0.050546            0.027354                 0.021099                      0.002618                       NaN                           NaN                  NaN                       NaN                      NaN                           NaN                  NaN                      NaN
    BEL             0.215385                  0.144169                  NaN                       NaN            

**Upstream Data Anomalies & Code Adjustments**: The data ingestion pipeline incorporates automated checks to handle structural data silences and reporting lags dynamically, ensuring mathematical validity:

1. **Data Isolation for the United States (USA) and China (CHN)**
   * **Anomaly:** Constant USD tracking for Manufacturing Value-Added (`NV.IND.MANF.KD`) in both countries contains no active time-series records except for an isolated base-year entry in **2015**.
   * **Pipeline Adjustment:** To prevent incomplete time series from distorting long-term trends, the processing script explicitly assigns `np.nan` to years with missing data. This safely isolates both countries from aggregate cross-country Compound Annual Growth Rate (CAGR) calculations, eliminating population bias.

2. **Reporting Lags and Dynamic Alignment for Japan (JPN)**
   * **Anomaly:** While Japan's real GDP is fully reported up to 2024, its industrial and manufacturing value-added indices suffer from an upstream lag, ending in **2023**.
   * **Pipeline Adjustment:** Rather than hardcoding fixed target years, the script implements a backward-scanning fallback function (`get_value_with_fallback`). For Japan, the pipeline detects 2023 as the latest valid year for manufacturing and **dynamically forces the GDP denominator to align with 2023**. This ensures strict temporal alignment (numerator and denominator are from the exact same year) and automatically truncates the CAGR horizon to 23 years (2000–2023) to preserve statistical integrity.

In [3]:

# 6. AUTOMATED ANALYSIS & INTERPRETATION
print("\n" + "="*50)
print("          DATA ANALYSIS INTERPRETATION          ")
print("="*50)

# Extract table column headers
mfg_share_col = [c for c in summary_table.columns if c.startswith("manufacturing_share_") and c not in [f"manufacturing_share_{start_year}", f"manufacturing_share_{end_year}"]][0]
ind_share_col = [c for c in summary_table.columns if c.startswith("industry_share_") and c not in [f"industry_share_{start_year}", f"industry_share_{end_year}"]][0]
gdp_cagr_col = [c for c in summary_table.columns if c.startswith("gdp_cagr_")][0]
ind_cagr_col = [c for c in summary_table.columns if c.startswith("industry_cagr_")][0]
mfg_cagr_col = [c for c in summary_table.columns if c.startswith("manufacturing_cagr_")][0]


# Q1: Which countries show signs of de‐industrialization?
# Methodology:
# (1) Primary Filter: Manufacturing share of GDP declined (Share Delta < 0)
# (2) Contextual Check: Did total industry share also decline?
# (3) Secondary Validation: Is manufacturing shrinking absolutely (CAGR < 0) or relatively (CAGR >= 0)?

mfg_share_changes = {}
ind_share_changes = {}

# Step 1: Calculate all deltas first
for _, r in summary_table.iterrows():
    country = r["country"]
    if pd.notna(r[f"manufacturing_share_{start_year}"]) and pd.notna(r[mfg_share_col]):
        mfg_share_changes[country] = r[mfg_share_col] - r[f"manufacturing_share_{start_year}"]
        
    if pd.notna(r[f"industry_share_{start_year}"]) and pd.notna(r[ind_share_col]):
        ind_share_changes[country] = r[ind_share_col] - r[f"industry_share_{start_year}"]

# Step 2: Apply primary filter (countries with declined manufacturing share)
mfg_change_series = pd.Series(mfg_share_changes)
de_ind_countries = mfg_change_series[mfg_change_series < 0].sort_values(ascending=True)

print("\n1. Countries with signs of de-industrialization (Manufacturing share declined):")

# Step 3: Extract values and determine logic for each de-industrializing country
for c_code, mfg_delta_val in de_ind_countries.items():
    
    c_row = summary_table[summary_table["country"] == c_code].iloc[0]
    c_mfg_cagr = c_row[mfg_cagr_col]
    ind_delta_val = ind_share_changes.get(c_code, float('nan'))
    
    # Determine logic (Relative vs. Absolute De-industrialization)
    if pd.notna(c_mfg_cagr):
        de_ind_type = "Absolute" if c_mfg_cagr < 0 else "Relative"
    else:
        de_ind_type = "Unclear"

    mfg_delta_str = f"{mfg_delta_val:.4%}"
    ind_delta_str = f"{ind_delta_val:.4%}" if pd.notna(ind_delta_val) else "NaN"
    c_mfg_cagr_str = f"{c_mfg_cagr:.4%}" if pd.notna(c_mfg_cagr) else "NaN"

    print(
        f"   - {c_code}: Industry Share Change: {ind_delta_str} | "
        f"Manufacturing Share Change: {mfg_delta_str} | "
        f"Manufacturing CAGR: {c_mfg_cagr_str} | Type: {de_ind_type}"
    )


# Q2: Is manufacturing declining faster than total industry?
# Methodology: Compare industry and manufacturing CAGRs globally and specifically within the de-industrializing group

# Global averages
avg_ind_cagr = summary_table[ind_cagr_col].mean()
avg_mfg_cagr = summary_table[mfg_cagr_col].mean()

# De-industrializing countries (identified in Q1)
de_ind_df = summary_table[summary_table["country"].isin(de_ind_countries.index)]
de_ind_avg_ind_cagr = de_ind_df[ind_cagr_col].mean()
de_ind_avg_mfg_cagr = de_ind_df[mfg_cagr_col].mean()

print("\n2. Is manufacturing declining faster than total industry?")
print(f"   - [Global Averages]             Industry CAGR: {avg_ind_cagr:.4%} | Manufacturing CAGR: {avg_mfg_cagr:.4%}")
print(f"   - [De-industrializing Group]   Industry CAGR: {de_ind_avg_ind_cagr:.4%} | Manufacturing CAGR: {de_ind_avg_mfg_cagr:.4%}")


# Q3: Are trends different between advanced and emerging economies?
# Methodology: Compare average CAGR between advanced and emerging countries

advanced_economies = ["AUS", "BEL", "CAN", "DEU", "DNK", "FIN", "FRA", "GBR", "ITA", "JPN", "USA"]
emerging_economies = ["BRA", "CHN", "MEX", "TUR", "ZAF"]

adv_df = summary_table[summary_table["country"].isin(advanced_economies)]
eme_df = summary_table[summary_table["country"].isin(emerging_economies)]

print("\n3. Macro Grouped Trends (Averages):")
print(f"   - Advanced Economies: GDP CAGR: {adv_df[gdp_cagr_col].mean():.4%} | Industry CAGR: {adv_df[ind_cagr_col].mean():.4%} | Manufacturing CAGR: {adv_df[mfg_cagr_col].mean():.4%}")
print(f"   - Emerging Economies: GDP CAGR: {eme_df[gdp_cagr_col].mean():.4%} | Industry CAGR: {eme_df[ind_cagr_col].mean():.4%} | Manufacturing CAGR: {eme_df[mfg_cagr_col].mean():.4%}")


          DATA ANALYSIS INTERPRETATION          

1. Countries with signs of de-industrialization (Manufacturing share declined):
   - CAN: Industry Share Change: -4.7607% | Manufacturing Share Change: -6.2613% | Manufacturing CAGR: -0.3716% | Type: Absolute
   - BRA: Industry Share Change: -4.5076% | Manufacturing Share Change: -4.3721% | Manufacturing CAGR: 0.6275% | Type: Relative
   - AUS: Industry Share Change: -3.2119% | Manufacturing Share Change: -4.0178% | Manufacturing CAGR: 0.2618% | Type: Relative
   - ZAF: Industry Share Change: -8.2838% | Manufacturing Share Change: -3.4799% | Manufacturing CAGR: 0.9657% | Type: Relative
   - BEL: Industry Share Change: -3.8515% | Manufacturing Share Change: -3.1836% | Manufacturing CAGR: 0.5167% | Type: Relative
   - FIN: Industry Share Change: -3.8313% | Manufacturing Share Change: -2.6874% | Manufacturing CAGR: 0.3553% | Type: Relative
   - MEX: Industry Share Change: -5.0078% | Manufacturing Share Change: -2.0860% | Manufacturing CAG

**Executive Summary of Analytical Findings**

**Q1: Which countries show signs of de‐industrialization?**
* De-industrialization is primarily identified by a structural decline in manufacturing value-added as a share of GDP ($\Delta < 0$), with manufacturing CAGR serving as a secondary validation metric:
* **Absolute De-industrialization:** **Canada** and **Italy** exhibit both a declining GDP share and a negative manufacturing CAGR ($-0.3716\%$ and $-0.0538\%$, respectively), signaling an actual contraction of real manufacturing output.
* **Relative De-industrialization:** **Brazil, Australia, South Africa, Belgium, Finland, Mexico, and France** display a declining manufacturing share despite positive manufacturing CAGRs. Their manufacturing sectors are expanding in absolute terms but are being outpaced and diluted by faster-growing domestic service sectors.

**Q2: Is manufacturing declining faster than total industry?**
* **Yes.** 
* Macroeconomic aggregates confirm that manufacturing growth lags behind broader industrial growth (which includes mining and construction). Globally, the average total industry CAGR ($1.5754\%$) outperforms the manufacturing CAGR ($1.2152\%$). Within the de-industrializing cohort, the gap persists (Industry CAGR: $0.8192\%$ vs. Manufacturing CAGR: $0.4486\%$), proving that manufacturing is the primary driver of industrial share decline.

**Q3: Are trends different between advanced and emerging economies?**
* **A clear macroeconomic divergence is observed.**
* **Emerging Economies** are undergoing rapid industrial expansion, posting strong growth across GDP (average CAGR: $3.7894\%$), industry, and manufacturing, solidifying their positions as global production hubs.
* **Advanced Economies** exhibit post-industrial consolidation characterized by low-speed growth (average GDP CAGR: $1.4115\%$) and stagnant manufacturing capacity, reflecting a mature structural shift toward high-value service sectors.